# Step 4 - Heterogeneous Effects & Qini Curves

**Ad Incrementality Analysis** (extends `analyze.py`, `naive.py`, `late.py`).

Steps 1-3 asked *how big* the ad effect is on average. This step asks
**"is the effect different across people, and who should we target?"**

We train an uplift model that predicts each user's individual treatment
effect, then draw a **Qini curve** showing how many extra conversions we
gain by targeting only the users the model ranks highest.

We run this for **both** targets: `visit` (common, stable) and
`conversion` (the real goal, but rare and noisier).

> Run the cells top to bottom. No GPU needed - uplift trees use CPU/RAM.

## 0. Settings

Tune these two values. Start light to confirm everything runs, then
increase for a heavier, more precise run. If the session runs out of
memory, lower `SAMPLE_ROWS`.

- Colab free tier has ~12-13 GB RAM. 3M rows + a medium model is a safe start.
- `MODEL_SIZE`: 'light' (fast), 'medium' (recommended), 'heavy' (slow).

In [ ]:
SAMPLE_ROWS = 3_000_000   # rows to sample (None = all 14M; risky on free tier)
MODEL_SIZE  = "medium"     # "light" | "medium" | "heavy"
RANDOM_SEED = 7

In [ ]:
!pip install scikit-uplift

## 1. Install libraries

scikit-uplift provides the uplift model and the Qini metric. We pin a
scikit-learn version it is compatible with to avoid warnings/errors.

In [ ]:
# scikit-uplift needs an older scikit-learn (its plotting calls a function
# removed in newer versions). We pin a compatible set and DO NOT use
# sklift.viz at all (we draw the Qini curve ourselves in cell 7).
!pip -q install "scikit-learn==1.3.2" scikit-uplift 2>/dev/null
print("Libraries ready.")

> **Important:** after the install cell finishes, go to
> **Runtime -> Restart session**, then continue from the next cell.
> This makes Colab actually use the pinned scikit-learn version.
> You do NOT need to re-run the install cell after restarting.

## 2. Download the data

Same public Criteo dataset as Steps 1-3 (~300 MB). Colab re-downloads it
each new session.

In [ ]:
import os, urllib.request

DATA_URL = ("https://huggingface.co/datasets/criteo/criteo-uplift/"
            "resolve/main/criteo-research-uplift-v2.1.csv.gz")
DATA_PATH = "criteo-uplift-v2.1.csv.gz"

if not os.path.exists(DATA_PATH):
    print("Downloading (~300 MB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Done.")
else:
    print("Already downloaded.")

## 3. Load & sample

We load the 12 features (f0-f11), the treatment flag, and both outcome
labels. If `SAMPLE_ROWS` is set, we take a random sample to fit in memory.

In [ ]:
import pandas as pd

feature_cols = [f"f{i}" for i in range(12)]
use_cols = feature_cols + ["treatment", "visit", "conversion"]

print("Loading data...")
df = pd.read_csv(DATA_PATH, usecols=use_cols)
print(f"Loaded {len(df):,} rows")

if SAMPLE_ROWS is not None and SAMPLE_ROWS < len(df):
    df = df.sample(n=SAMPLE_ROWS, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Sampled down to {len(df):,} rows")

## 4. Set up the model

We use a **Two-Model** uplift approach: train one model on the treated
group and one on the control group, then the predicted uplift for a user
is the difference in their predicted outcome probability.

Model size maps to how many trees / how deep - bigger = more precise but
slower.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklift.models import TwoModels

SIZE = {
    "light":  dict(n_estimators=50,  max_depth=8),
    "medium": dict(n_estimators=150, max_depth=12),
    "heavy":  dict(n_estimators=400, max_depth=None),
}[MODEL_SIZE]
print("Model config:", SIZE)

def make_model():
    return TwoModels(
        estimator_trmnt=RandomForestClassifier(
            n_jobs=-1, random_state=RANDOM_SEED, **SIZE),
        estimator_ctrl=RandomForestClassifier(
            n_jobs=-1, random_state=RANDOM_SEED, **SIZE),
        method="vanilla",
    )

## 5. Train & evaluate for one target

This function splits the data into train/test, fits the uplift model,
predicts each test user's uplift, and computes the **Qini AUC score**
(higher = the model is better at finding responsive users).

In [ ]:
from sklearn.model_selection import train_test_split
from sklift.metrics import qini_auc_score

def run_for_target(target_name):
    X = df[feature_cols]
    y = df[target_name]
    t = df["treatment"]

    X_tr, X_te, y_tr, y_te, t_tr, t_te = train_test_split(
        X, y, t, test_size=0.3, random_state=RANDOM_SEED)

    print(f"[{target_name}] training on {len(X_tr):,} rows...")
    model = make_model()
    model.fit(X_tr, y_tr, t_tr)

    uplift = model.predict(X_te)
    score = qini_auc_score(y_te, uplift, t_te)
    print(f"[{target_name}] Qini AUC = {score:.4f}")
    return dict(target=target_name, y_te=y_te.to_numpy(),
                uplift=uplift, t_te=t_te.to_numpy(), score=score)

## 6. Run for BOTH targets and compare

Trains two models (visit and conversion). This is the slow cell - a few
minutes on 'medium', longer on 'heavy'.

In [ ]:
results = {}
for target in ["visit", "conversion"]:
    results[target] = run_for_target(target)

print("\nQini AUC summary")
for target, r in results.items():
    print(f"  {target:11s}: {r['score']:.4f}")

## 7. Qini curves side by side

The Qini curve shows: as we target more users (x-axis, ranked by predicted
uplift), how many *incremental* conversions we accumulate (y-axis). A curve
that bulges above the diagonal means the model successfully concentrates the
ad effect in the top-ranked users - i.e. targeting works.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def qini_curve(y_true, uplift, treatment, n_points=100):
    """Return (x, y) points of the Qini curve, without sklift.viz."""
    y_true = np.asarray(y_true); uplift = np.asarray(uplift)
    treatment = np.asarray(treatment)
    order = np.argsort(uplift)[::-1]      # best-predicted uplift first
    y = y_true[order]; t = treatment[order]
    n_t = np.cumsum(t); n_c = np.cumsum(1 - t)
    n_t = np.where(n_t == 0, 1, n_t); n_c = np.where(n_c == 0, 1, n_c)
    cum_y_t = np.cumsum(y * t); cum_y_c = np.cumsum(y * (1 - t))
    qini = cum_y_t - cum_y_c * (n_t / n_c)   # incremental outcomes
    frac = np.arange(1, len(y) + 1) / len(y)
    idx = np.linspace(0, len(y) - 1, n_points).astype(int)
    return frac[idx], qini[idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, target in zip(axes, ["visit", "conversion"]):
    r = results[target]
    x, q = qini_curve(r["y_te"], r["uplift"], r["t_te"])
    ax.plot(x, q, label="model", color="#185FA5")
    ax.plot([0, 1], [0, q[-1]], "--", color="gray", label="random")
    ax.set_xlabel("fraction of users targeted (ranked by predicted uplift)")
    ax.set_ylabel("cumulative incremental outcomes")
    ax.set_title(f"Qini curve - {target}  (AUC={r['score']:.4f})")
    ax.legend()
plt.tight_layout()
plt.savefig("qini_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: qini_curves.png")

## 8. Interpretation

- A **higher Qini AUC** and a curve bulging above the diagonal mean the ad
  effect is **heterogeneous** - some users are far more responsive, and the
  model can find them.
- `visit` is usually the cleaner, higher curve (visits are ~16x more common
  than conversions, so the signal is stronger).
- `conversion` is noisier because conversions are rare (~0.3%); a weaker
  curve here is expected, not a bug.

**Takeaway for the project:** beyond knowing the *average* ad effect
(Steps 1-3), targeting the users the model ranks highest yields more
incremental outcomes per ad shown - the practical payoff of incrementality
analysis.